Metrics (1)

- mean_mdl — mean dependency length

In [1]:
!python -m spacy download en_core_web_trf

/Users/adil.shamim/Desktop/Academic/code/mtech-thesis/mtech-thesis-project/thesis-venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
     - 0 bytes ? 0:00:000m
ERROR: Wheel 'en-core-web-trf' located at /private/var/folders/ps/ddq35s1529g2lj9_07cbv2480000gp/T/pip-unpack-v6fpxqfi/en_core_web_trf-3.8.0-py3-none-any.whl is invalid.


In [1]:
import spacy

# en_core_web_trf (RoBERTa based parser, LAS 0.938) - more accurate parser than en_core_web_sm (LAS 0.899)
# MDL needs only pos_ (tagger + attribute_ruler) and dep_/head/sents (parser), so ner and lemmatizer are disabled
spacy.prefer_gpu() # uses the Mac GPU (MPS) if available
nlp = spacy.load("en_core_web_trf", disable=["ner", "lemmatizer"])
print(f"Language model: {nlp.meta['lang']}_{nlp.meta['name']}")
print(f"Version: {nlp.meta['version']}")
print(f"Pipe names: {nlp.pipe_names}")

/Users/adil.shamim/Desktop/Academic/code/mtech-thesis/mtech-thesis-project/thesis-venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/adil.shamim/Desktop/Academic/code/mtech-thesis/mtech-thesis-project/thesis-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Language model: en_core_web_trf
Version: 3.8.0
Pipe names: ['transformer', 'tagger', 'parser', 'attribute_ruler']


In [11]:
def get_sentences(doc):
    return list(doc.sents)

In [12]:
def sentence_mdl(sent):
    #prefix array to find number of punctuation tokens up to any index of this sentence
    punct_token_indices = [0] * len(sent)
    num_punct_indices_seen = 0
    for index, token in enumerate(sent):
        if token.pos_ == "PUNCT":
           num_punct_indices_seen += 1
        punct_token_indices[index] = num_punct_indices_seen

    dependency_lengths = [] #list to store dependency lengths of this sentence
    for token in sent:
        if token.dep_ == "ROOT" or token.pos_ == "PUNCT":
           continue
        head = token.head
        #if head is punctuation, walk up to the nearest non-punctuation ancestor
        while head.pos_ == "PUNCT" and head.i != head.head.i:
            head = head.head
        if head.pos_ == "PUNCT":
           continue
        raw_distance = abs(head.i - token.i)
        #token.i is index in whole doc, so subtract sent.start to get index inside the sentence
        punct_between = abs(punct_token_indices[head.i - sent.start] - punct_token_indices[token.i - sent.start])
        length = raw_distance - punct_between
        dependency_lengths.append(length)

    if not dependency_lengths: #no arcs (single word / only punctuation), so no MDL for this sentence
       return None
    return sum(dependency_lengths) / len(dependency_lengths)

In [13]:
from spacy.tokens import Doc

def calculate_mean_mdl(text):
    #text can be a raw string or an already parsed doc (from nlp.pipe), so the paragraph is never parsed twice
    doc = text if isinstance(text, Doc) else nlp(text)
    sentence_mdls = [] #list to store mean dependency length of each sentence
    for sent in get_sentences(doc):
        mdl = sentence_mdl(sent)
        if mdl is not None: #skip sentences with no arcs
           sentence_mdls.append(mdl)

    #paragraph MDL = unweighted mean of sentence MDLs
    if sentence_mdls:
        mean_dependency_length = sum(sentence_mdls) / len(sentence_mdls)
    else:
        mean_dependency_length = 0

    return f"{mean_dependency_length:.5f}"

In [16]:
from pathlib import Path
import csv

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/raw-csv to read and data/<LEVEL>/features/ to write
raw_csv_path = Path(f"../../data/{LEVEL}/raw-csv")
# paragraph: demo/ keeps a demo run away from the real data/paragraph/features/mdl-csv (switch it for a real re-run)
out_csv_path = Path("../../data/paragraph/demo/mdl-csv") if LEVEL == "paragraph" else Path(f"../../data/{LEVEL}/features/mdl-csv")
out_csv_path.mkdir(parents=True, exist_ok=True)

for csv_file in sorted(raw_csv_path.glob("*.csv")):
    out_csv_file = out_csv_path / csv_file.name
    if out_csv_file.exists() and out_csv_file.stat().st_size > 0: # Skip if already exist
       continue
    print(f"Working on file: {csv_file.name}")
    with csv_file.open("r",encoding="utf-8") as cf:
       csv_reader = csv.DictReader(cf)
       column_names = list(csv_reader.fieldnames)
       column_names.insert(-1, "mean_mdl")
       rows = list(csv_reader)
    texts = [row["text"] for row in rows]
    # write to a temp file and rename at the end, so a half written file is never skipped on restart
    tmp_csv_file = out_csv_file.with_suffix(".csv.tmp")
    with tmp_csv_file.open("w",encoding="utf-8") as of:
       csv_writer = csv.DictWriter(of,fieldnames=column_names)
       csv_writer.writeheader()
       # nlp.pipe parses texts in batches - much faster than calling nlp(text) for every row
       for row, doc in zip(rows, nlp.pipe(texts, batch_size=64)):
          row["mean_mdl"] = calculate_mean_mdl(doc)
          csv_writer.writerow(row)
    tmp_csv_file.rename(out_csv_file)
    #break #- During demo we can uncomment and providea demo path


Working on file: 00s-Music-Quiz.csv


In [15]:
# TEST MDL CODE

text = "Beyoncé and Rihanna are tied for the second - most #1 entries on the Billboard Hot 100 chart, with 5 each."
print(calculate_mean_mdl(text))
text = "Timbaland was one of the most successful producers of the 2000s. His biggest hits include “Promiscuous” by Nelly Furtado, “Work It” by Missy Elliott, and “Cry Me a River” by Justin Timberlake."
print(calculate_mean_mdl(text))

# per sentence check - paragraph MDL must be the mean of the sentence MDLs (same doc, no re-parse)
doc = nlp(text)
sentence_values = [sentence_mdl(sent) for sent in get_sentences(doc)]
print(sentence_values)
assert abs(float(calculate_mean_mdl(doc)) - sum(sentence_values) / len(sentence_values)) < 1e-4

# edge cases - no arcs must give 0, not a division by zero
assert calculate_mean_mdl("") == "0.00000"
assert calculate_mean_mdl("!!!") == "0.00000"
assert calculate_mean_mdl("Hello.") == "0.00000"
print("all MDL checks passed")

2.95000
1.85000
[1.6, 2.1]
all MDL checks passed


In [14]:

# DEMO - HOW SENTENCE LEVEL MDL WORKS
# step 1: split paragraph into sentences (dependency tree is defined per sentence)
# step 2: for every sentence, find length of each dependency arc and take mean = sentence MDL
# step 3: paragraph MDL = unweighted mean of sentence MDLs

demo_text = "Timbaland was one of the most successful producers of the 2000s. His biggest hits include “Promiscuous” by Nelly Furtado, “Work It” by Missy Elliott, and “Cry Me a River” by Justin Timberlake."
demo_doc = nlp(demo_text)

headings = ["DEPENDENT", "DEP_I", "HEAD", "HEAD_I", "RAW", "PUNCT", "LENGTH", "NOTE"]
demo_sentence_mdls = [] #list to store mean dependency length of each sentence

for sent_number, sent in enumerate(get_sentences(demo_doc), start=1):
    #prefix array to find number of punctuation tokens up to any index of this sentence
    punct_token_indices = [0] * len(sent)
    num_punct_indices_seen = 0
    for index, token in enumerate(sent):
        if token.pos_ == "PUNCT":
           num_punct_indices_seen += 1
        punct_token_indices[index] = num_punct_indices_seen

    print(f"\n=== Sentence {sent_number}: {sent.text.strip()}")
    print(f"{headings[0]:15} {headings[1]:6} {headings[2]:15} {headings[3]:7} {headings[4]:5} {headings[5]:6} {headings[6]:7} {headings[7]}")

    dependency_lengths = [] #list to store dependency lengths of this sentence
    for token in sent:
        if token.dep_ == "ROOT": #root has no head, so no arc
           print(f"{token.text:15} {token.i:<6} {'-':15} {'-':7} {'-':5} {'-':6} {'-':7} skipped (ROOT)")
           continue
        if token.pos_ == "PUNCT": #punctuation is not counted as a word
           print(f"{token.text:15} {token.i:<6} {'-':15} {'-':7} {'-':5} {'-':6} {'-':7} skipped (PUNCT)")
           continue
        head = token.head
        note = ""
        #if head is punctuation, walk up to the nearest non-punctuation ancestor
        while head.pos_ == "PUNCT" and head.i != head.head.i:
            head = head.head
            note = f"head was punct '{token.head.text}', moved up"
        if head.pos_ == "PUNCT":
           print(f"{token.text:15} {token.i:<6} {head.text:15} {head.i:<7} {'-':5} {'-':6} {'-':7} skipped (no non-punct head)")
           continue
        raw_distance = abs(head.i - token.i)
        #token.i is index in whole doc, so subtract sent.start to get index inside the sentence
        punct_between = abs(punct_token_indices[head.i - sent.start] - punct_token_indices[token.i - sent.start])
        length = raw_distance - punct_between
        dependency_lengths.append(length)
        print(f"{token.text:15} {token.i:<6} {head.text:15} {head.i:<7} {raw_distance:<5} {punct_between:<6} {length:<7} {note}")

    if dependency_lengths:
        sent_mdl = sum(dependency_lengths) / len(dependency_lengths)
        demo_sentence_mdls.append(sent_mdl)
        print(f"Sentence {sent_number} MDL = {sum(dependency_lengths)} / {len(dependency_lengths)} = {sent_mdl:.4f}")
        assert abs(sent_mdl - sentence_mdl(sent)) < 1e-9 #same value as the real function
    else:
        print(f"Sentence {sent_number} has no arcs, so it is skipped")

#paragraph MDL = unweighted mean of sentence MDLs (no divide by zero if no sentence has arcs)
if demo_sentence_mdls:
    paragraph_mdl = sum(demo_sentence_mdls) / len(demo_sentence_mdls)
else:
    paragraph_mdl = 0
print(f"\nParagraph MDL = mean of {[round(m, 4) for m in demo_sentence_mdls]} = {paragraph_mdl:.5f}")
print(f"calculate_mean_mdl gives: {calculate_mean_mdl(demo_doc)}") #same value as the real function



=== Sentence 1: Timbaland was one of the most successful producers of the 2000s.
DEPENDENT       DEP_I  HEAD            HEAD_I  RAW   PUNCT  LENGTH  NOTE
Timbaland       0      was             1       1     0      1       
was             1      -               -       -     -      -       skipped (ROOT)
one             2      was             1       1     0      1       
of              3      one             2       1     0      1       
the             4      producers       7       3     0      3       
most            5      successful      6       1     0      1       
successful      6      producers       7       1     0      1       
producers       7      of              3       4     0      4       
of              8      producers       7       1     0      1       
the             9      2000s           10      1     0      1       
2000s           10     of              8       2     0      2       
.               11     -               -       -     -      -       skip